# EDA & Tiền xử lý — Phát hiện bất thường KHÔNG giám sát (T1.csv)

Notebook phân tích dữ liệu SCADA tuabin gió thật (`T1.csv`, năm 2018, bước 10 phút). Dữ liệu **không có nhãn** nên đây là bài toán phát hiện bất thường **không giám sát**: ta mô hình hóa trạng thái "bình thường" (đường cong công suất theo tốc độ gió) và đánh dấu các điểm lệch mạnh.

Hai module dùng lại:
1. `preprocessing.py` — xử lý missing values, outliers, chuẩn hóa, chia train/test theo thời gian.


In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, sys

# --- Tự dò PROJECT_ROOT chứa src/ và data/ ---
_p = os.getcwd()
for _ in range(6):
    if os.path.isdir(os.path.join(_p, 'src')) and os.path.isdir(os.path.join(_p, 'data')):
        break
    _p = os.path.dirname(_p)
PROJECT_ROOT = _p
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
RAW_DIR = os.path.join(PROJECT_ROOT, 'data', 'raw')
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
FEATURES_DIR = os.path.join(PROJECT_ROOT, 'data', 'features')
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(FEATURES_DIR, exist_ok=True)
sys.path.insert(0, SRC_DIR)

import preprocessing as prep
import feature_engineering as feat

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('Đã nạp thư viện & module thành công!')

ModuleNotFoundError: No module named 'pandas'

## 1. Đọc dữ liệu & khảo sát ban đầu


In [ ]:
csv_path = os.path.join(RAW_DIR, 'T1.csv')
df = prep.load_data(csv_path)
print('Kích thước sau khi reindex lưới thời gian:', df.shape)
df.head()

: 

: 

In [ ]:
df.info()

: 

: 

In [ ]:
df.describe().round(2)

: 

: 

## 2. Chất lượng dữ liệu


In [ ]:
num_cols = ['LV ActivePower (kW)', 'Wind Speed (m/s)', 'Theoretical_Power_Curve (KWh)', 'Wind Direction (°)']
print('Khoảng thời gian:', df['timestamp'].min(), '->', df['timestamp'].max())
print('Tổng số mốc (lưới đều):', len(df))
print('\nSố giá trị khuyết mỗi cột (do mốc thời gian bị thiếu):')
print(df[num_cols].isnull().sum())
print(f"\nTỷ lệ khuyết: {df[num_cols].isnull().any(axis=1).mean()*100:.2f}%")

: 

: 

## 3. Biến động tín hiệu theo thời gian

In [ ]:
fig, axes = plt.subplots(len(num_cols), 1, figsize=(15, 14), sharex=True)
for i, col in enumerate(num_cols):
    axes[i].plot(df['timestamp'], df[col], color='#4c72b0', alpha=0.8, linewidth=0.5)
    axes[i].set_ylabel(col, fontsize=10, fontweight='bold')
    axes[i].grid(True, linestyle='--', alpha=0.5)
plt.xlabel('Thời gian')
plt.suptitle('Tín hiệu SCADA tuabin gió năm 2018', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout(); plt.show()

: 

: 

## 4. Phân phối tần suất các biến

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10)); axes = axes.flatten()
for i, col in enumerate(num_cols):
    sns.histplot(data=df, x=col, kde=True, ax=axes[i], color='teal', bins=60)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
plt.tight_layout(); plt.show()

: 

: 

## 5. Ma trận tương quan

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_cols].corr(), annot=True, cmap='coolwarm', fmt='.3f', vmin=-1, vmax=1, linewidths=0.5)
plt.title('Tương quan tuyến tính (Pearson)', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

: 

: 

## 6. Phân tích đường cong công suất (Power Curve)


In [ ]:
plt.figure(figsize=(10, 7))
plt.scatter(df['Wind Speed (m/s)'], df['LV ActivePower (kW)'], s=3, alpha=0.2, label='Thực tế (ActivePower)')
order = df['Wind Speed (m/s)'].argsort()
plt.plot(df['Wind Speed (m/s)'].iloc[order], df['Theoretical_Power_Curve (KWh)'].iloc[order],
         color='red', linewidth=2, label='Đường cong lý thuyết')
plt.xlabel('Tốc độ gió (m/s)'); plt.ylabel('Công suất (kW)')
plt.title('Đường cong công suất: Thực tế vs Lý thuyết', fontsize=13, fontweight='bold')
plt.legend(); plt.tight_layout(); plt.show()

: 

: 

## 7. Phần dư (residual) — tín hiệu bất thường


In [ ]:
df_r = prep.handle_missing_values(df, columns=num_cols, strategy='interpolate')
df_r['power_residual'] = df_r['LV ActivePower (kW)'] - df_r['Theoretical_Power_Curve (KWh)']
mu, sigma = df_r['power_residual'].mean(), df_r['power_residual'].std()
df_r['residual_z'] = (df_r['power_residual'] - mu) / sigma
df_r['anomaly_candidate'] = (df_r['residual_z'].abs() > 3).astype(int)
print(f"Số điểm ứng viên bất thường (|z|>3): {int(df_r['anomaly_candidate'].sum())} "
      f"({df_r['anomaly_candidate'].mean()*100:.2f}%)")

fig, axes = plt.subplots(2, 1, figsize=(15, 9))
sns.histplot(df_r['power_residual'], bins=80, ax=axes[0], color='slateblue')
axes[0].set_title('Phân phối phần dư công suất', fontweight='bold')
axes[0].axvline(mu+3*sigma, color='r', ls='--'); axes[0].axvline(mu-3*sigma, color='r', ls='--')
axes[1].plot(df_r['timestamp'], df_r['power_residual'], lw=0.4, color='gray', label='residual')
anom = df_r[df_r['anomaly_candidate']==1]
axes[1].scatter(anom['timestamp'], anom['power_residual'], color='red', s=8, label='ứng viên bất thường')
axes[1].set_title('Phần dư theo thời gian & điểm bất thường', fontweight='bold'); axes[1].legend()
plt.tight_layout(); plt.show()

: 

: 

## 8. Tiền xử lý: missing & outliers


In [ ]:
print('NaN trước khi xử lý:', int(df[num_cols].isnull().sum().sum()))
df_clean = prep.handle_missing_values(df, columns=num_cols, strategy='interpolate')
print('NaN sau interpolate:', int(df_clean[num_cols].isnull().sum().sum()))

: 

: 

In [ ]:
# --- CẬP NHẬT: TÍCH HỢP PIPELINE MỚI ---
# Các hàm IQR/Z-score cũ đã bị xóa. Thay bằng các hàm vật lý và tọa độ mới.
df_clean = prep.clean_physical_limits(df_clean)
df_clean = prep.encode_wind_direction(df_clean)
df_clean = prep.extract_time_features(df_clean)
df_clean = prep.create_labels(df_clean, loss_threshold=0.5)

print('Đã hoàn tất Pipeline tiền xử lý Vật lý & Lượng giác!')
display(df_clean.head())

: 

: 

## 9. Kỹ thuật đặc trưng


In [ ]:
fe_cols = ['LV ActivePower (kW)', 'Wind Speed (m/s)', 'Theoretical_Power_Curve (KWh)']
df_clean['power_residual'] = df_clean['LV ActivePower (kW)'] - df_clean['Theoretical_Power_Curve (KWh)']

df_features = feat.calculate_rolling_stats(df_clean, columns=fe_cols, windows=[6, 24])
df_features = feat.calculate_z_scores(df_features, columns=fe_cols)
df_features = feat.calculate_differences(df_features, columns=fe_cols, periods=[1])
print('Số cột ban đầu:', df_clean.shape[1], '-> sau feature engineering:', df_features.shape[1])
new_cols = [c for c in df_features.columns if c not in df_clean.columns]
print(f'{len(new_cols)} đặc trưng mới, ví dụ:', new_cols[:8])

: 

: 

## 10. Chia train/test theo thời gian & chuẩn hóa


In [ ]:
train_df, test_df = prep.split_train_test_chrono(df_features, test_size=0.3)
feature_cols = [c for c in df_features.columns if c != 'timestamp']
train_df, scale_stats = prep.scale_features(train_df, columns=feature_cols, method='standard', return_stats=True)
test_df = prep.scale_features(test_df, columns=feature_cols, method='standard', stats=scale_stats)
print(f'Train: {train_df.shape[0]} mẫu | Test: {test_df.shape[0]} mẫu')

train_df.to_csv(os.path.join(FEATURES_DIR, 'T1_train.csv'), index=False)
test_df.to_csv(os.path.join(FEATURES_DIR, 'T1_test.csv'), index=False)
print('Đã xuất T1_train.csv, T1_test.csv vào data/features/')

: 

: 

## 11. Xuất bảng đặc trưng đầy đủ (chưa scale)

In [ ]:
out = os.path.join(PROCESSED_DIR, 'T1_processed.csv')
df_features.to_csv(out, index=False)
print('Đã xuất:', out, '|', df_features.shape)

: 

: 

## Kết luận & hướng tiếp theo
- Dữ liệu thật có gap thời gian (~2030 mốc thiếu) đã được xử lý bằng reindex + nội suy.
- Phân tích đường cong công suất & phần dư cho tín hiệu bất thường rõ ràng mà **không cần nhãn**.
- Đặc trưng (rolling/z-score/diff + residual) đã sẵn sàng cho mô hình **không giám sát**.

